# Linear Regression vs. Local LLM + RAG

Exam comparison exercise: predict the **same numerical target** with two different approaches and compare the errors.

**Task:** predict the sale price of an apartment (in thousand DKK) from its size (in m²).
The data is synthetic and generated with a fixed seed, so anyone can reproduce the results.

**Rules from the exam (the notebook follows all of them):**
- Both approaches get the same test inputs, and neither sees the test targets.
- The LLM's RAG collection only contains background text and examples from the **training** rows.
- The LLM never calls the regression model or runs regression code.
- We report measured results, not assumed ones.

**Before you run it:** Ollama must be running with the chat model and the embedding model pulled
(`ollama pull qwen3:4b` and `ollama pull nomic-embed-text`).

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

OLLAMA = "http://localhost:11434"
CHAT_MODEL = "qwen3:4b"
EMBED_MODEL = "nomic-embed-text"

SEED = 42        # change it to see how much the results depend on the data
N_TEST = 5       # number of held-out test cases
TOP_K = 5        # chunks retrieved per question
RAG_DIR = Path("rag_data")
RAG_DIR.mkdir(exist_ok=True)

## Step 1: Prepare the data

- **Input:** `size_m2`, the apartment size in square metres.
- **Target:** `price_kdkk`, the sale price in thousand DKK.
- The data is generated from a straight line plus random noise, so a linear model is the right tool for it.
- A few rows are held out as **test cases**. Their prices are stored separately and are not used until the comparison step.

In [ ]:
rng = np.random.default_rng(SEED)
n = 45
size_m2 = rng.integers(30, 151, size=n)
price = 42 * size_m2 + 350 + rng.normal(0, 120, size=n)
df = pd.DataFrame({"size_m2": size_m2, "price_kdkk": price.round(0).astype(int)})

train_df, test_df = train_test_split(df, test_size=N_TEST, random_state=SEED)
train_df = train_df.sort_values("size_m2").reset_index(drop=True)
test_df = test_df.sort_values("size_m2").reset_index(drop=True)

# From here on, the models only get the test INPUTS.
# The test targets are kept in y_test and are only read in the comparison step.
X_test = test_df[["size_m2"]]
y_test = test_df["price_kdkk"]

print(f"{len(train_df)} training rows, {len(test_df)} test cases")
print("Test inputs (m2):", X_test["size_m2"].tolist())
train_df.head()

## Step 2: Train the linear regression model

Fit on the training rows only, then predict the held-out inputs.

In [ ]:
model = LinearRegression()
model.fit(train_df[["size_m2"]], train_df["price_kdkk"])
reg_pred = model.predict(X_test)

print(f"Learned line: price = {model.coef_[0]:.2f} * size + {model.intercept_:.2f}")
print("Regression predictions:", reg_pred.round(0).tolist())

## Step 3: Build the RAG collection for the local LLM

What the LLM is allowed to retrieve from:
- one short background note, and
- one sentence per **training** row.

Design choice (worth explaining in the synopsis): each training row is its own chunk.
Cutting the data by character count, like the starter script does, could split a row in half.

Nothing from the test set goes into the collection. The collection is also saved to `rag_data/rag_collection.txt`
so you can include it in the submission and show exactly what the LLM had access to.

In [ ]:
background = (
    "Background: this dataset describes apartment sales. The sale price is given in thousand DKK "
    "and the size in square metres. In general, larger apartments sell for higher prices."
)
chunks = [background] + [
    f"An apartment of {row.size_m2} m2 sold for {row.price_kdkk} thousand DKK."
    for row in train_df.itertuples()
]

(RAG_DIR / "rag_collection.txt").write_text("\n".join(chunks), encoding="utf-8")
print(len(chunks), "chunks in the RAG collection")
for c in chunks[:3]:
    print("-", c)

### Helper functions

The same building blocks as the starter script: embeddings, cosine similarity retrieval and a chat call.
`temperature` is set to 0 so the model answers as consistently as possible between runs.

In [ ]:
def embed(texts):
    r = requests.post(
        f"{OLLAMA}/api/embed",
        json={"model": EMBED_MODEL, "input": texts},
        timeout=300,
    )
    r.raise_for_status()
    return np.array(r.json()["embeddings"], dtype=np.float32)


def cosine_top_k(query_vec, matrix, k):
    q = query_vec / np.linalg.norm(query_vec)
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    scores = m @ q
    top = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top]


def strip_thinking(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()


def chat(system, user):
    if "qwen3" in CHAT_MODEL.lower():
        user += "\n\n/no_think"
    r = requests.post(
        f"{OLLAMA}/api/chat",
        json={
            "model": CHAT_MODEL,
            "stream": False,
            "think": False,
            "options": {"temperature": 0},
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        },
        timeout=600,
    )
    r.raise_for_status()
    return strip_thinking(r.json()["message"]["content"])


def parse_number(text):
    """Take the first number in the model's answer (drops thousands separators)."""
    match = re.search(r"\d[\d,]*\.?\d*", text)
    return float(match.group().replace(",", "")) if match else float("nan")


chunk_vectors = embed(chunks)
print("Embedded", chunk_vectors.shape[0], "chunks, vector size", chunk_vectors.shape[1])

## Step 4: Let the local LLM predict the test cases

The prompt uses the 4T's: **Traits**, **Task**, **Tone** and **Target**.
Here the *target* is a script that reads the answer, so the model is told to reply with one number only.

Each test case is predicted twice:
- **With RAG:** the most similar chunks are retrieved and pasted into the prompt.
- **Without RAG:** the same question with no context. This is an extra baseline that helps answer
  "was RAG useful?".

The regression model is never called here.

In [ ]:
SYSTEM_PROMPT = """\
Traits: You are a careful data analyst who estimates numbers from the examples you are given.
Task: Predict the sale price of an apartment from its size, using the example sales in the context.
Tone: Precise and matter-of-fact. No explanations.
Target: A script that reads your answer. Reply with ONLY one number: the price in thousand DKK, no unit and no text.
"""

NO_RAG_PROMPT = SYSTEM_PROMPT.replace(
    "using the example sales in the context", "using your general knowledge"
)


def make_question(size):
    return f"Predict the sale price (in thousand DKK) of an apartment of {size} m2."


def predict_with_rag(size):
    question = make_question(size)
    hits = cosine_top_k(embed([question])[0], chunk_vectors, TOP_K)
    context = "\n".join(chunks[i] for i, _ in hits)
    answer = chat(SYSTEM_PROMPT, f"Context:\n{context}\n\n{question}")
    return parse_number(answer), answer, hits


def predict_without_rag(size):
    answer = chat(NO_RAG_PROMPT, make_question(size))
    return parse_number(answer), answer


rag_pred, no_rag_pred, retrieval_log = [], [], []
for size in X_test["size_m2"]:
    p_rag, raw, hits = predict_with_rag(size)
    p_plain, _ = predict_without_rag(size)
    rag_pred.append(p_rag)
    no_rag_pred.append(p_plain)
    retrieval_log.append((size, raw, [(round(s, 3), chunks[i]) for i, s in hits]))
    print(f"{size} m2 -> with RAG: {p_rag}, without RAG: {p_plain}")

### What was retrieved?

The exam asks you to document what the LLM retrieved. Look at this output and ask:
did retrieval return examples with *similar sizes*, or just any rows?
Embeddings capture meaning, so they may not be good at "closeness of numbers".

In [ ]:
for size, raw, hits in retrieval_log:
    print(f"\n=== {size} m2 | raw answer: {raw!r}")
    for score, text in hits:
        print(f"  [{score}] {text}")

## Step 5: Compare the results

Only now are the real test prices used. The error measure is the **absolute error** per case,
`|prediction - actual|`, and the **mean absolute error (MAE)** across all test cases.
Units are thousand DKK.

In [ ]:
results = pd.DataFrame({
    "size_m2": X_test["size_m2"],
    "actual": y_test,
    "regression": reg_pred.round(0),
    "llm_rag": rag_pred,
    "llm_no_rag": no_rag_pred,
})
for col in ["regression", "llm_rag", "llm_no_rag"]:
    results[f"abs_err_{col}"] = (results[col] - results["actual"]).abs().round(0)

error_cols = [c for c in results.columns if c.startswith("abs_err_")]
mae = results[error_cols].mean().round(1)

display(results)
print("Mean absolute error (thousand DKK):")
print(mae.to_string())
if results[["llm_rag", "llm_no_rag"]].isna().any().any():
    print("\nWarning: some LLM answers had no number in them (NaN). Check the raw answers above.")

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(train_df["size_m2"], train_df["price_kdkk"], color="lightgray", label="training data")
xs = pd.DataFrame({"size_m2": np.linspace(df["size_m2"].min(), df["size_m2"].max(), 50)})
plt.plot(xs["size_m2"], model.predict(xs), label="linear regression")
plt.scatter(results["size_m2"], results["actual"], color="black", label="actual (test)")
plt.scatter(results["size_m2"], results["llm_rag"], marker="x", color="tab:red", label="LLM + RAG")
plt.scatter(results["size_m2"], results["llm_no_rag"], marker="+", color="tab:orange", label="LLM without RAG")
plt.xlabel("size (m2)")
plt.ylabel("price (thousand DKK)")
plt.legend()
plt.tight_layout()
plt.show()

## Step 6: Reflect (answer these in the synopsis, with your own measured numbers)

1. **Which approach had the lower MAE, and by how much?** Quote the numbers from the table above.
2. **What did retrieval return?** Were the retrieved examples the ones with the closest sizes?
   Do embeddings capture numeric closeness?
3. **What did the LLM actually do?** Copy a nearby example, interpolate, or guess?
4. **Was RAG useful?** Compare `llm_rag` with `llm_no_rag`.
5. **When is a trained model better than an LLM?** Think about exactness, speed, cost, repeatability and explainability.
6. **Limitations you saw** (small model, tiny dataset, one test split, answer parsing).

### Optional experiments (change one thing at a time and write down the result)
- `TOP_K` = 1, 5 and the full collection (`len(chunks)`).
- A different `SEED`, or more test cases (`N_TEST`).
- Remove the background note from `chunks`.
- Another chat model, for example a smaller or bigger one.

Remember: the aim is to compare approaches and explain the differences, not to make the LLM win.